# NB7 — LOO diagnosis V1 on frozen V5

**Scope:** validation negatives only; the test split is never loaded.

The canonical scorer remains locked to 3–8 items. For an original 3-item outfit, LOO creates 2-item subsets. This notebook uses the explicit **eval-only** `diagnostic_min_items=2` path and reports those rows separately as extrapolation. It does not retrain or modify `best.pt`.

In [ ]:
from pathlib import Path
import json
import os
import subprocess
import sys

REPO_URL = "https://github.com/ThinhTran2208/opisoverated.git"
BRANCH = "feat/diagnosis-loo-v1"
REPO_ROOT = Path("/content/opisoverated")

if not REPO_ROOT.exists():
    subprocess.run(
        ["git", "clone", "--branch", BRANCH, "--single-branch", REPO_URL, str(REPO_ROOT)],
        check=True,
    )
else:
    subprocess.run(["git", "-C", str(REPO_ROOT), "fetch", "origin", BRANCH], check=True)
    subprocess.run(["git", "-C", str(REPO_ROOT), "checkout", BRANCH], check=True)
    subprocess.run(["git", "-C", str(REPO_ROOT), "pull", "--ff-only", "origin", BRANCH], check=True)

if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from google.colab import drive
drive.mount("/content/drive")

ARTIFACT_ROOT = Path("/content/drive/MyDrive/ML_Final")
os.environ["FASHION_ARTIFACT_ROOT"] = str(ARTIFACT_ROOT)
os.environ["FASHION_EMBEDDING_CACHE"] = str(ARTIFACT_ROOT / "fashionclip_item_embeddings.pt")
os.environ["FASHION_EMBEDDING_MANIFEST"] = str(ARTIFACT_ROOT / "embedding_manifest_v1.json")
os.environ["FASHION_CORE7_DIR"] = str(ARTIFACT_ROOT / "polyvore_core7_v2" / "core7_drop_v2")
os.environ["FASHION_SCORER_READY_DIR"] = str(ARTIFACT_ROOT / "polyvore_core7_v2" / "scorer_ready_v2")

subprocess.run([sys.executable, "-m", "pip", "install", "-q", "pyyaml"], check=True)

HEAD = subprocess.check_output(
    ["git", "-C", str(REPO_ROOT), "rev-parse", "HEAD"], text=True
).strip()
STATUS = subprocess.check_output(
    ["git", "-C", str(REPO_ROOT), "status", "--porcelain"], text=True
).strip()
assert not STATUS, f"Repository must be clean, got: {STATUS}"
print("Branch  :", BRANCH)
print("Git HEAD:", HEAD)


In [ ]:
# Run the new LOO tests plus the existing scorer regression tests.
for pattern in ("test_loo*.py", "test_scorer*.py"):
    test_run = subprocess.run(
        [sys.executable, "-m", "unittest", "discover", "-s", "tests", "-p", pattern, "-v"],
        cwd=REPO_ROOT,
        text=True,
        capture_output=True,
    )
    print(test_run.stdout)
    if test_run.stderr:
        print(test_run.stderr)
    if test_run.returncode != 0:
        raise RuntimeError(f"Tests failed for {pattern}: {test_run.returncode}")
print("LOO + SCORER REGRESSION TESTS: PASS")


In [ ]:
import torch

from src.data.runtime_paths import load_runtime_paths
from src.diagnosis.loo import diagnose_outfit, evaluate_loo_localization
from src.scorer.checkpoint import build_runtime_provenance, load_checkpoint
from src.scorer.dataset import EmbeddingStore, build_dataset_from_runtime
from src.scorer.model import TypeAwarePairwiseScorer

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
paths = load_runtime_paths(repo_root=REPO_ROOT)
provenance = build_runtime_provenance(paths, REPO_ROOT)

# Load only validation data. Train and test are intentionally not constructed.
embedding_store = EmbeddingStore(paths.embedding_cache)
valid_dataset = build_dataset_from_runtime(
    paths, "valid", embedding_store=embedding_store
)
assert len(valid_dataset) == 2284

BEST_PATH = (
    REPO_ROOT
    / "artifacts"
    / "checkpoints"
    / "type_aware_pairwise_v1"
    / "final_val_auc_v5_seed42"
    / "best.pt"
)
assert BEST_PATH.is_file(), f"Missing canonical checkpoint: {BEST_PATH}"

payload = load_checkpoint(
    BEST_PATH,
    map_location="cpu",
    current_provenance=provenance,
)
model = TypeAwarePairwiseScorer.from_config(payload["config"])
model.load_state_dict(payload["model_state_dict"])
model.to(device).eval()

assert model.min_items == 3
assert payload["epoch"] == 52
assert abs(float(payload["best_valid_roc_auc"]) - 0.6905082489625538) < 1e-12

print("Device              :", device)
print("Validation samples  :", len(valid_dataset))
print("Paired families     :", len(valid_dataset.pair_families))
print("Checkpoint epoch    :", payload["epoch"])
print("Checkpoint valid AUC:", payload["best_valid_roc_auc"])
print("Canonical min_items :", model.min_items)
print("CHECKPOINT + VALIDATION DATA: PASS")


In [ ]:
# Smoke-test one real validation negative before running the full metric.
negative_index = next(
    index
    for index, row in enumerate(valid_dataset.records)
    if row["label"] == 0
)
sample = valid_dataset[negative_index]
smoke = diagnose_outfit(
    model,
    sample["item_embeddings"],
    sample["coarse_category_ids"],
    item_ids=sample["item_ids"],
)
smoke["target_swapped_item_index"] = sample["negative_metadata"]["swapped_item_index"]
print(json.dumps(smoke, indent=2, ensure_ascii=False))


In [ ]:
# Full validation-negative LOO evaluation. Originals + all removals are batched.
report = evaluate_loo_localization(
    model,
    valid_dataset,
    outfit_batch_size=64,
)
assert report["overall"]["sample_count"] == 1142

report["split"] = "valid"
report["git_head"] = HEAD
report["checkpoint"] = {
    "path": str(BEST_PATH.relative_to(REPO_ROOT)),
    "epoch": int(payload["epoch"]),
    "best_valid_roc_auc": float(payload["best_valid_roc_auc"]),
}

summary = {
    "protocol_version": report["protocol_version"],
    "split": report["split"],
    "overall": report["overall"],
    "by_original_item_count": report["by_original_item_count"],
}
print(json.dumps(summary, indent=2, ensure_ascii=False))

RUN_DIR = ARTIFACT_ROOT / "diagnosis_runs" / "loo_diagnostic_v1_v5_seed42"
RUN_DIR.mkdir(parents=True, exist_ok=True)
REPORT_PATH = RUN_DIR / "validation_loo_report.json"
REPORT_PATH.write_text(
    json.dumps(report, indent=2, ensure_ascii=False), encoding="utf-8"
)
print("Saved:", REPORT_PATH)
print("TEST SPLIT WAS NOT LOADED.")


In [ ]:
# Inspect a few failures. Size-3 rows used two-item extrapolation.
failures = [row for row in report["records"] if not row["top1_correct"]]
print("Incorrect cases:", len(failures))
for row in failures[:10]:
    print(
        row["sample_id"],
        "n=", row["original_item_count"],
        "target=", row["target_swapped_item_index"],
        "pred=", row["predicted_problematic_item_index"],
        "2-item-extrapolation=", row["uses_two_item_extrapolation"],
        "deltas=", [round(value, 4) for value in row["deltas_without_minus_full"]],
    )
